In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# Step 1: Load standard dataset
categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.space'
]

data = fetch_20newsgroups(
    subset='train',
    categories=categories,
    remove=('headers', 'footers', 'quotes')
)

documents = data.data
actual_labels = data.target

# Step 2: Convert documents into TF-IDF vectors
vectorizer = TfidfVectorizer(
    stop_words='english',
    max_features=5000
)

X = vectorizer.fit_transform(documents)

# Step 3: Apply K-Means clustering
k = 3

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

cluster_labels = kmeans.fit_predict(X)

# Step 4: Map each cluster to the actual class
# Find the most common actual class in each cluster
cluster_to_class = {}

for cluster in range(k):
    indices = np.where(cluster_labels == cluster)[0]

    if len(indices) > 0:
        labels = actual_labels[indices]
        majority_class = np.bincount(labels).argmax()
        cluster_to_class[cluster] = majority_class

# Convert cluster labels to predicted class labels
predicted_labels = np.array([
    cluster_to_class[c] for c in cluster_labels
])

# Step 5: Calculate Purity
def calculate_purity(actual, predicted_clusters):
    total = len(actual)
    correct = 0

    for cluster in np.unique(predicted_clusters):
        indices = np.where(predicted_clusters == cluster)[0]
        labels = actual[indices]

        if len(labels) > 0:
            correct += np.bincount(labels).max()

    return correct / total

purity = calculate_purity(actual_labels, cluster_labels)

# Step 6: Calculate Precision, Recall and F-measure
precision = precision_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

recall = recall_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

f_measure = f1_score(
    actual_labels,
    predicted_labels,
    average='macro',
    zero_division=0
)

# Step 7: Display results
print("Number of Documents:", len(documents))
print("Number of Features:", X.shape[1])
print("Number of Clusters:", k)

print("\nPerformance Measures:")
print("Purity    :", round(purity, 4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F-measure :", round(f_measure, 4))

Number of Documents: 1657
Number of Features: 5000
Number of Clusters: 3

Performance Measures:
Purity    : 0.717
Precision : 0.808
Recall    : 0.7346
F-measure : 0.7145


In [2]:
# IRS LAB - EXPERIMENT 5
# Text Document Clustering Using K-Means

import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.optimize import linear_sum_assignment

# 1. Load 20 Newsgroups Dataset
categories = ['comp.graphics', 'rec.sport.baseball',
              'sci.space', 'talk.politics.misc']

data = fetch_20newsgroups(
    subset='all', categories=categories,
    remove=('headers', 'footers', 'quotes'))

# 2. TF-IDF Representation
vectorizer = TfidfVectorizer(stop_words='english',
                             max_features=10000)
X = vectorizer.fit_transform(data.data)

print("Number of Documents:", len(data.data))
print("Vocabulary Size:", len(vectorizer.vocabulary_))

# 3. K-Means Clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X)

print("\nK-Means Clustering Completed")
print("Cluster Distribution:", np.bincount(clusters))

# 4. Contingency Matrix
cm = confusion_matrix(data.target, clusters)
print("\nContingency Matrix:")
print(cm)

# 5. Hungarian Algorithm - Cluster Mapping
row, col = linear_sum_assignment(-cm)
mapping = {c: r for r, c in zip(row, col)}

mapped = np.array([mapping[c] for c in clusters])

print("\nCluster to Class Mapping:")
for c, r in mapping.items():
    print("Cluster", c, "->", data.target_names[r])

# 6. Purity
purity = np.sum(np.max(cm, axis=0)) / len(data.target)

# 7. Precision, Recall and F-Measure
precision = precision_score(data.target, mapped,
                            average='weighted', zero_division=0)
recall = recall_score(data.target, mapped,
                      average='weighted', zero_division=0)
fmeasure = f1_score(data.target, mapped,
                    average='weighted', zero_division=0)

# 8. ARI and NMI
ari = adjusted_rand_score(data.target, clusters)
nmi = normalized_mutual_info_score(data.target, clusters)

print("\nPerformance Measures:")
print("Purity    :", round(purity, 4))
print("Precision :", round(precision, 4))
print("Recall    :", round(recall, 4))
print("F-Measure :", round(fmeasure, 4))
print("ARI       :", round(ari, 4))
print("NMI       :", round(nmi, 4))

# 9. Classification Report
print("\nClassification Report:")
print(classification_report(
    data.target, mapped,
    target_names=data.target_names,
    zero_division=0))

# 10. Confusion Matrix After Mapping
print("Confusion Matrix After Mapping:")
print(confusion_matrix(data.target, mapped))

Number of Documents: 3729
Vocabulary Size: 10000

K-Means Clustering Completed
Cluster Distribution: [ 689 1993  593  454]

Contingency Matrix:
[[630 333   2   8]
 [ 29 379 586   0]
 [ 25 516   2 444]
 [  5 765   3   2]]

Cluster to Class Mapping:
Cluster 0 -> comp.graphics
Cluster 2 -> rec.sport.baseball
Cluster 3 -> sci.space
Cluster 1 -> talk.politics.misc

Performance Measures:
Purity    : 0.6503
Precision : 0.8406
Recall    : 0.6503
F-Measure : 0.6727
ARI       : 0.2607
NMI       : 0.4458

Classification Report:
                    precision    recall  f1-score   support

     comp.graphics       0.91      0.65      0.76       973
rec.sport.baseball       0.99      0.59      0.74       994
         sci.space       0.98      0.45      0.62       987
talk.politics.misc       0.38      0.99      0.55       775

          accuracy                           0.65      3729
         macro avg       0.82      0.67      0.67      3729
      weighted avg       0.84      0.65      0.67      